# 3N — Neural Stream: Weekly State Graph Preprocessing

## Objective

Prepare weekly state-level graph signals for the DCRNN. This is intentionally different from Stream I:

- The DCRNN has one node for every `State` category found in the complete source-derived panel, including `__UNKNOWN__` if needed.
- Every report counted by Notebook 1 is represented in exactly one state/week cell.
- The graph is a state-adjacency graph, not the former 5 km grid with a top-128-node truncation.
- Weekly target counts are preserved per state. The DCRNN predicts each state's next-week count, and Notebook 4N sums all node predictions to obtain the national forecast.

No scaler is fitted here. Notebook 4N fits log-count normalization separately inside each training fold, and once again on the complete 80% training period for the final model.

## 0. Load the shared complete weekly state panel

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

import matplotlib.pyplot as plt

panel_path = resolve_artifact('weekly_state_panel.parquet')
week_path = resolve_artifact('weekly_index.parquet')
split_path = resolve_artifact('split_metadata.json')
panel = pd.read_parquet(panel_path)
week_index = pd.read_parquet(week_path).sort_values('week_id')
split_meta = json.loads(split_path.read_text())
TRAIN_END_WEEK_ID = int(split_meta['train_end_week_id'])
STATES = sorted(panel['state'].astype(str).unique().tolist())
WEEK_IDS = week_index['week_id'].to_numpy(dtype=np.int32)
WEEK_DATES = pd.to_datetime(week_index['week_start']).to_numpy()
T = len(WEEK_IDS)
N = len(STATES)

counts_df = panel.pivot_table(index='week_id', columns='state', values='accident_count', aggfunc='sum', fill_value=0)
counts_df = counts_df.reindex(index=WEEK_IDS, columns=STATES, fill_value=0)
Y_COUNTS = counts_df.to_numpy(dtype=np.float32)
if Y_COUNTS.shape != (T, N):
    raise AssertionError(f'Unexpected state count tensor shape: {Y_COUNTS.shape}')
if not np.allclose(Y_COUNTS.sum(axis=1),
                   panel.groupby('week_id').accident_count.sum().reindex(WEEK_IDS).to_numpy()):
    raise AssertionError('State count tensor does not conserve national weekly totals.')
print('Time steps:', T, '| graph nodes:', N, '| train end week_id:', TRAIN_END_WEEK_ID)
print('Observed state labels:', STATES)

## 1. Construct the state adjacency graph

Edges are undirected state-border adjacency relationships. The graph includes self-loops. States not present in the data are not added. If the source contains an unrecognized state label, it remains a node with a self-loop rather than being dropped; the adjacency audit will make this explicit.

In [ ]:
# Neighbour relationships for US states and DC. Symmetry is enforced below.
STATE_NEIGHBORS = {
'AL':['FL','GA','MS','TN'], 'AZ':['CA','NV','NM','UT'], 'AR':['LA','MO','MS','OK','TN','TX'],
'CA':['AZ','NV','OR'], 'CO':['KS','NE','NM','OK','UT','WY'], 'CT':['MA','NY','RI'],
'DE':['MD','NJ','PA'], 'DC':['MD','VA'], 'FL':['AL','GA'], 'GA':['AL','FL','NC','SC','TN'],
'ID':['MT','NV','OR','UT','WA','WY'], 'IL':['IA','IN','KY','MO','WI'], 'IN':['IL','KY','MI','OH'],
'IA':['IL','MN','MO','NE','SD','WI'], 'KS':['CO','MO','NE','OK'], 'KY':['IL','IN','MO','OH','TN','VA','WV'],
'LA':['AR','MS','TX'], 'ME':['NH'], 'MD':['DC','DE','PA','VA','WV'], 'MA':['CT','NH','NY','RI','VT'],
'MI':['IN','OH','WI'], 'MN':['IA','ND','SD','WI'], 'MS':['AL','AR','LA','TN'],
'MO':['AR','IA','IL','KS','KY','NE','OK','TN'], 'MT':['ID','ND','SD','WY'],
'NE':['CO','IA','KS','MO','SD','WY'], 'NV':['AZ','CA','ID','OR','UT'], 'NH':['ME','MA','VT'],
'NJ':['DE','NY','PA'], 'NM':['AZ','CO','OK','TX'], 'NY':['CT','MA','NJ','PA','VT'],
'NC':['GA','SC','TN','VA'], 'ND':['MN','MT','SD'], 'OH':['IN','KY','MI','PA','WV'],
'OK':['AR','CO','KS','MO','NM','TX'], 'OR':['CA','ID','NV','WA'], 'PA':['DE','MD','NJ','NY','OH','WV'],
'RI':['CT','MA'], 'SC':['GA','NC'], 'SD':['IA','MN','MT','ND','NE','WY'],
'TN':['AL','AR','GA','KY','MO','MS','NC','VA'], 'TX':['AR','LA','NM','OK'],
'UT':['AZ','CO','ID','NV','WY'], 'VT':['MA','NH','NY'], 'VA':['DC','KY','MD','NC','TN','WV'],
'WA':['ID','OR'], 'WV':['KY','MD','OH','PA','VA'], 'WI':['IA','IL','MI','MN'],
'WY':['CO','ID','MT','NE','SD','UT'], 'AK':[], 'HI':[],
}

state_to_idx = {s:i for i,s in enumerate(STATES)}
adj = np.zeros((N, N), dtype=np.float32)
for i, state in enumerate(STATES):
    adj[i, i] = 1.0
    neighbours = STATE_NEIGHBORS.get(state, [])
    for neighbour in neighbours:
        if neighbour in state_to_idx:
            j = state_to_idx[neighbour]
            adj[i, j] = 1.0
            adj[j, i] = 1.0

# Any non-standard / unknown node remains in the tensor and receives a self-loop only.
degree_no_self = (adj - np.eye(N, dtype=np.float32)).sum(axis=1)
isolated = [STATES[i] for i in range(N) if degree_no_self[i] == 0]
edge_count = int(np.count_nonzero(np.triu(adj, k=1)))
print('Undirected adjacency edges:', edge_count)
print('Self-loop-only nodes:', isolated)

np.save(ARTIFACT_DIR / 'dcrnn_adjacency.npy', adj)
(ARTIFACT_DIR / 'dcrnn_state_nodes.json').write_text(json.dumps({
    'states': STATES, 'state_to_node_id': state_to_idx,
    'self_loop_only_nodes': isolated,
    'graph': 'undirected state-border adjacency + self loops',
}, indent=2))

plt.figure(figsize=(12, 4))
plt.bar(STATES, degree_no_self)
plt.title('State graph degree (excluding self-loops)')
plt.xlabel('State node'); plt.ylabel('Adjacent observed states')
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 2. Create weekly calendar and state-identity covariates

For every state and week, future-known covariates are:

- sine and cosine of ISO week-of-year;
- a linear `week_id` trend scaled by the end of the training period;
- a one-hot state identity vector.

No weather value from an accident-only event row is used as a future covariate. Weather measured only when accidents were reported would be missing systematically in many zero-count state-weeks.

In [ ]:
week_sin = []
week_cos = []
trend = []
for idx, date_value in enumerate(WEEK_DATES):
    d = pd.Timestamp(date_value)
    iso_week = int(d.isocalendar().week)
    week_sin.append(np.sin(2*np.pi*iso_week/52.1775))
    week_cos.append(np.cos(2*np.pi*iso_week/52.1775))
    trend.append(float(WEEK_IDS[idx] / max(TRAIN_END_WEEK_ID, 1)))
calendar = np.column_stack([week_sin, week_cos, trend]).astype(np.float32)
state_identity = np.eye(N, dtype=np.float32)

# Broadcast calendar and one-hot state identity to [time, nodes, covariates].
calendar_by_node = np.broadcast_to(calendar[:, None, :], (T, N, calendar.shape[1]))
identity_by_time = np.broadcast_to(state_identity[None, :, :], (T, N, N))
C_RAW = np.concatenate([calendar_by_node, identity_by_time], axis=-1).astype(np.float32)

np.save(ARTIFACT_DIR / 'dcrnn_weekly_state_counts.npy', Y_COUNTS)
np.save(ARTIFACT_DIR / 'dcrnn_future_covariates.npy', C_RAW)
np.save(ARTIFACT_DIR / 'dcrnn_week_ids.npy', WEEK_IDS)
np.save(ARTIFACT_DIR / 'dcrnn_week_dates.npy', WEEK_DATES.astype('datetime64[ns]'))

metadata = {
    'frequency': 'weekly', 'forecast_horizon': 1,
    'n_time_steps': T, 'n_nodes': N, 'state_nodes': STATES,
    'train_end_week_id': TRAIN_END_WEEK_ID,
    'first_test_week_id': TRAIN_END_WEEK_ID + 1,
    'target': 'per-state weekly accident report count; national forecast is sum over all nodes',
    'covariates': ['week_of_year_sin','week_of_year_cos','week_id_scaled'] + [f'state_onehot_{s}' for s in STATES],
    'graph': 'undirected state-border adjacency with self-loops; no event/node truncation',
    'target_scaling': 'not fitted here; fit on each training fold in 4N',
    'week_definition': split_meta['week_definition'],
}
(ARTIFACT_DIR / 'dcrnn_weekly_preprocessing_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Counts shape:', Y_COUNTS.shape, '| covariates shape:', C_RAW.shape, '| adjacency shape:', adj.shape)

## 3. Validate row conservation and split boundaries

In [ ]:
assert np.all(Y_COUNTS >= 0)
assert np.isfinite(Y_COUNTS).all() and np.isfinite(C_RAW).all()
assert adj.shape == (N, N) and np.allclose(adj, adj.T)
assert np.all(np.diag(adj) > 0)
train_mask = WEEK_IDS <= TRAIN_END_WEEK_ID
test_mask = WEEK_IDS > TRAIN_END_WEEK_ID
assert train_mask.sum() + test_mask.sum() == T
print('Weekly graph preprocessing checks passed.')